In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import mlflow
import mlflow.pytorch

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

c:\Users\sgfar\anaconda3\envs\torch311\Lib\site-packages\mlflow\protos\service_pb2.py:11: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  from google.protobuf import service as _service
c:\Users\sgfar\anaconda3\envs\torch311\Lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


In [2]:
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"Device:          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

CUDA available:  True
Device:          NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
df = pd.read_csv("../data/attack_data.csv")

print(f"Dataframe Attack: {df.shape[0]} rows x {df.shape[1]} columns.")
display(df.head())
print(f"\nLabel distribution:\n{df['attack_label'].value_counts()}")

Dataframe Attack: 210 rows x 5 columns.


,distance_to_player,angle_to_player,enemy_speed,player_speed,attack_label
0,175.9032,2.8730,0.0,382.7462,2
1,179.4757,9.6998,0.0,500.0000,2
2,137.6861,13.4604,0.0,440.4138,0
3,69.1040,12.5118,0.0,121.6102,0
4,177.8203,2.2851,0.0,378.1962,2



Label distribution:
attack_label
2    116
0     94
Name: count, dtype: int64


In [5]:
#Remapping labels
df["attack_label"] = df["attack_label"].map({0:0,2:1})

print("=== After remapping ===")
print(df['attack_label'].value_counts())

=== After remapping ===
attack_label
1    116
0     94
Name: count, dtype: int64


In [6]:
df.head()

,distance_to_player,angle_to_player,enemy_speed,player_speed,attack_label
0,175.9032,2.8730,0.0,382.7462,1
1,179.4757,9.6998,0.0,500.0000,1
2,137.6861,13.4604,0.0,440.4138,0
3,69.1040,12.5118,0.0,121.6102,0
4,177.8203,2.2851,0.0,378.1962,1


In [7]:
X = df.drop(labels=["attack_label","enemy_speed"],axis=1).values
y = df["attack_label"].values

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=SEED)

print(f"Train size: {X_train.shape[0]} samples")
print(f"Test  size: {X_test.shape[0]} samples")
print(f"\nTrain label distribution: {np.bincount(y_train)}")
print(f"Test  label distribution: {np.bincount(y_test)}")

Train size: 168 samples
Test  size: 42 samples

Train label distribution: [74 94]
Test  label distribution: [20 22]


In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

feature_names = ['distance_to_player', 'angle_to_player', 'player_speed']
for i, name in enumerate(feature_names):
    print(f"  {name}:")
    print(f"    mean = {scaler.mean_[i]:.4f}")
    print(f"    std  = {scaler.scale_[i]:.4f}")

print(f"\nX_train_scaled sample (first row):")
print(f"  Before: {X_train[0]}")
print(f"  After:  {X_train_scaled[0].round(4)}")

  distance_to_player:
    mean = 145.3633
    std  = 41.4379
  angle_to_player:
    mean = 8.0484
    std  = 7.4728
  player_speed:
    mean = 336.1864
    std  = 193.3947

X_train_scaled sample (first row):
  Before: [178.7268   4.7559 365.6658]
  After:  [ 0.8051 -0.4406  0.1524]


In [9]:
class ACDataset(Dataset):
    def __init__(self,X,y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        return self.X[idx],self.y[idx]

In [10]:
BATCH_SIZE = 32

train_dataset = ACDataset(X_train_scaled,y_train)
test_dataset = ACDataset(X_test_scaled,y_test)

train_dataloader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_dataloader  = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Test  dataset: {len(test_dataset)} samples")
print(f"Train batches: {len(train_dataloader)}")
print(f"\nSample batch shapes:")
for X_batch, y_batch in train_dataloader:
    print(f"  X_batch: {X_batch.shape}")
    print(f"  y_batch: {y_batch.shape}")
    break

Train dataset: 168 samples
Test  dataset: 42 samples
Train batches: 6

Sample batch shapes:
  X_batch: torch.Size([32, 3])
  y_batch: torch.Size([32])


In [11]:
train_dataset.__getitem__(0)

(tensor([ 0.8051, -0.4406,  0.1524]), tensor(1))

In [12]:
import os
os.makedirs("../models",exist_ok=True)

with open('../models/attack_scaler.pkl','wb') as f:
    pickle.dump(scaler,f)

print("Scaler saved to ../models/attack_scaler.pkl")
print(f"Scaler params saved: mean={scaler.mean_}, scale={scaler.scale_}")

Scaler saved to ../models/attack_scaler.pkl
Scaler params saved: mean=[145.3632881    8.04843274 336.18638631], scale=[ 41.43793973   7.47275915 193.39474181]
